# Practico Parcial 1 — Canal BSC y regla de decision ML

## El problema en una oración

Se transmite un bit $A$ a través de un canal que puede "flipear" (cambiar) ese bit con cierta probabilidad. Se recibe un bit $Y$. La pregunta es: **¿cuál es la mejor forma de adivinar qué se transmitió, mirando solo lo que llegó?**

---

## Modelo del sistema

El símbolo transmitido es binario:

$$A \in \{0, 1\}$$

El símbolo observado (recibido) también es binario:

$$Y \in \{0, 1\}$$

Las **probabilidades a priori** dicen cuán probable es que se transmita cada bit *antes* de observar nada:

$$P_A(0) = q \qquad \text{(probabilidad de transmitir 0)}$$

$$P_A(1) = 1 - q \qquad \text{(probabilidad de transmitir 1)}$$

El canal BSC tiene una única regla de comportamiento:

- Con probabilidad $1-p$: el bit llega **igual** a como se transmitió.
- Con probabilidad $p$: el bit llega **cambiado** (se produjo un error de canal).

Eso genera exactamente cuatro probabilidades condicionales:

$$P_{Y|A}(0|0) = 1-p \qquad \text{(se transmitió 0, llegó 0, sin error)}$$

$$P_{Y|A}(1|0) = p \qquad \text{(se transmitió 0, llegó 1, hubo error)}$$

$$P_{Y|A}(0|1) = p \qquad \text{(se transmitió 1, llegó 0, hubo error)}$$

$$P_{Y|A}(1|1) = 1-p \qquad \text{(se transmitió 1, llegó 1, sin error)}$$

Estas probabilidades son las **verosimilitudes**: dicen cuán probable es cada observación dado cada posible símbolo transmitido.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8-whitegrid")
rng = np.random.default_rng(20260507)

## 1. Tabla de probabilidades condicionadas del BSC

El BSC tiene dos entradas posibles y dos salidas posibles.

Si se transmite $A=0$:

- se observa $Y=0$ con probabilidad $1-p$;
- se observa $Y=1$ con probabilidad $p$.

Si se transmite $A=1$:

- se observa $Y=0$ con probabilidad $p$;
- se observa $Y=1$ con probabilidad $1-p$.

Tabla:

| Observacion $y$ | $P_{Y|A}(y|0)$ | $P_{Y|A}(y|1)$ |
|---:|---:|---:|
| 0 | $1-p$ | $p$ |
| 1 | $p$ | $1-p$ |

Esta tabla contiene las **verosimilitudes**, que son las probabilidades que usa la regla ML.


In [ ]:
def bsc_likelihoods(p):
    y_values = np.array([0, 1])
    py_given_a0 = np.array([1 - p, p])
    py_given_a1 = np.array([p, 1 - p])
    return y_values, py_given_a0, py_given_a1

p = 0.3
y_values, py_given_a0, py_given_a1 = bsc_likelihoods(p)

print("Tabla del BSC para p = 0.3")
print("y   P(Y=y|A=0)   P(Y=y|A=1)")
for y, p0, p1 in zip(y_values, py_given_a0, py_given_a1):
    print(f"{y}       {p0:.1f}           {p1:.1f}")

## 2. Regla de decision ML general

La regla ML elige el valor de $A$ que hace mas probable la observacion recibida $Y=y$.

$$\hat A_{ML}(y)=\arg\max_{a\in\{0,1\}} P_{Y|A}(y|a)$$

Como el problema es binario, para cada valor observado comparamos:

$$P_{Y|A}(y|0) \quad \text{contra} \quad P_{Y|A}(y|1)$$

La regla ML no multiplica por $P_A(a)$. Por eso, la regla ML no usa el valor de $q$.

### Decision cuando se observa $y=0$

Comparamos:

$$P_{Y|A}(0|0)=1-p$$

$$P_{Y|A}(0|1)=p$$

Si $p<0.5$, entonces:

$$1-p>p$$

por lo tanto:

$$\hat A_{ML}(0)=0$$

### Decision cuando se observa $y=1$

Comparamos:

$$P_{Y|A}(1|0)=p$$

$$P_{Y|A}(1|1)=1-p$$

Si $p<0.5$, entonces:

$$p<1-p$$

por lo tanto:

$$\hat A_{ML}(1)=1$$

Entonces, para $p<0.5$, la regla ML es:

$$\boxed{\hat A_{ML}=Y}$$

Es decir: si el canal se equivoca con probabilidad menor que $0.5$, lo mas verosimil es que no haya cambiado el bit.


## 3. Punto 1: regla ML para $q=0.6$ y $p=0.3$

Datos:

$$q=0.6$$

$$P_A(0)=q=0.6$$

$$P_A(1)=1-q=1-0.6=0.4$$

$$p=0.3$$

$$1-p=1-0.3=0.7$$

Para ML usamos solamente las probabilidades condicionadas $P_{Y|A}(y|a)$.

### Si se observa $y=0$

$$P_{Y|A}(0|0)=1-p=0.7$$

$$P_{Y|A}(0|1)=p=0.3$$

Comparamos:

$$0.7>0.3$$

Entonces:

$$\boxed{\hat A_{ML}(0)=0}$$

### Si se observa $y=1$

$$P_{Y|A}(1|0)=p=0.3$$

$$P_{Y|A}(1|1)=1-p=0.7$$

Comparamos:

$$0.3<0.7$$

Entonces:

$$\boxed{\hat A_{ML}(1)=1}$$

Respuesta del punto 1:

$$\boxed{\hat A_{ML}=Y}$$

Observacion importante: el valor $q=0.6$ no cambia la regla ML, porque ML no usa probabilidades a priori.


## 4. Punto 2: regla ML para $q=0.5$ y $p=0.3$

Datos:

$$q=0.5$$

$$P_A(0)=q=0.5$$

$$P_A(1)=1-q=1-0.5=0.5$$

$$p=0.3$$

$$1-p=1-0.3=0.7$$

De nuevo, para ML usamos solamente las probabilidades condicionadas.

### Si se observa $y=0$

$$P_{Y|A}(0|0)=1-p=0.7$$

$$P_{Y|A}(0|1)=p=0.3$$

Comparamos:

$$0.7>0.3$$

Entonces:

$$\boxed{\hat A_{ML}(0)=0}$$

### Si se observa $y=1$

$$P_{Y|A}(1|0)=p=0.3$$

$$P_{Y|A}(1|1)=1-p=0.7$$

Comparamos:

$$0.3<0.7$$

Entonces:

$$\boxed{\hat A_{ML}(1)=1}$$

Respuesta del punto 2:

$$\boxed{\hat A_{ML}=Y}$$

Como antes, el resultado ML es igual porque depende de $p$, no de $q$.


In [ ]:
width = 0.35
x = np.arange(len(y_values))

plt.figure(figsize=(8, 4))
plt.bar(x - width/2, py_given_a0, width, label=r"$P_{Y|A}(y|0)$")
plt.bar(x + width/2, py_given_a1, width, label=r"$P_{Y|A}(y|1)$")
plt.xticks(x, y_values)
plt.xlabel("y")
plt.ylabel("Probabilidad")
plt.title("Verosimilitudes del BSC para p = 0.3")
plt.legend()
plt.show()

## 5. Punto 3: es optima la regla ML?

Para saber si una regla minimiza la probabilidad de error, el criterio correcto es MAP.

La regla MAP decide:

$$\hat A_{MAP}(y)=\arg\max_{a\in\{0,1\}} P_{A|Y}(a|y)$$

Usando Bayes:

$$P_{A|Y}(a|y)=\frac{P_A(a)P_{Y|A}(y|a)}{P_Y(y)}$$

Como $P_Y(y)$ no depende de $a$, maximizar $P_{A|Y}(a|y)$ equivale a maximizar:

$$P_A(a)P_{Y|A}(y|a)$$

Entonces, para comparar con ML, revisamos si las decisiones ML coinciden con las decisiones MAP.

Si coinciden, la regla ML tambien es optima para minimizar $P_e$.

Si no coinciden, la regla ML no es optima y la que minimiza $P_e$ es MAP.


## 6. Optimalidad para el punto 1: $q=0.6$, $p=0.3$

Datos:

$$P_A(0)=0.6, \qquad P_A(1)=0.4$$

$$p=0.3, \qquad 1-p=0.7$$

### Si se observa $y=0$

MAP compara:

$$P_A(0)P_{Y|A}(0|0)$$

contra:

$$P_A(1)P_{Y|A}(0|1)$$

Reemplazando:

$$P_A(0)P_{Y|A}(0|0)=0.6\cdot 0.7$$

$$P_A(0)P_{Y|A}(0|0)=0.42$$

$$P_A(1)P_{Y|A}(0|1)=0.4\cdot 0.3$$

$$P_A(1)P_{Y|A}(0|1)=0.12$$

Comparamos:

$$0.42>0.12$$

Entonces MAP decide:

$$\hat A_{MAP}(0)=0$$

Esta decision coincide con ML.

### Si se observa $y=1$

MAP compara:

$$P_A(0)P_{Y|A}(1|0)$$

contra:

$$P_A(1)P_{Y|A}(1|1)$$

Reemplazando:

$$P_A(0)P_{Y|A}(1|0)=0.6\cdot 0.3$$

$$P_A(0)P_{Y|A}(1|0)=0.18$$

$$P_A(1)P_{Y|A}(1|1)=0.4\cdot 0.7$$

$$P_A(1)P_{Y|A}(1|1)=0.28$$

Comparamos:

$$0.18<0.28$$

Entonces MAP decide:

$$\hat A_{MAP}(1)=1$$

Esta decision tambien coincide con ML.

Conclusion para el punto 1:

$$\boxed{\hat A_{ML}=\hat A_{MAP}=Y}$$

Por lo tanto, para $q=0.6$ y $p=0.3$, la regla ML si es optima.


## 7. Optimalidad para el punto 2: $q=0.5$, $p=0.3$

Datos:

$$P_A(0)=0.5, \qquad P_A(1)=0.5$$

$$p=0.3, \qquad 1-p=0.7$$

Cuando las hipotesis son equiprobables:

$$P_A(0)=P_A(1)$$

la regla MAP se reduce a ML, porque el factor a priori es el mismo para las dos hipotesis.

Lo vemos paso a paso.

MAP compara:

$$P_A(0)P_{Y|A}(y|0)$$

contra:

$$P_A(1)P_{Y|A}(y|1)$$

Como:

$$P_A(0)=P_A(1)=0.5$$

queda comparar:

$$0.5P_{Y|A}(y|0)$$

contra:

$$0.5P_{Y|A}(y|1)$$

Como el factor $0.5$ multiplica ambos lados, no cambia la decision. Entonces MAP y ML toman la misma decision.

Conclusion para el punto 2:

$$\boxed{\hat A_{ML}=\hat A_{MAP}=Y}$$

Por lo tanto, para $q=0.5$ y $p=0.3$, la regla ML si es optima.


## 8. Probabilidad de error de la regla ML en los puntos 1 y 2

En ambos puntos, para $p=0.3$, la regla ML queda:

$$\hat A=Y$$

La regla se equivoca cuando el canal cambia el bit.

La probabilidad de error general es:

$$P_e=P_A(0)P(\hat A=1|A=0)+P_A(1)P(\hat A=0|A=1)$$

Con la regla $\hat A=Y$:

$$P(\hat A=1|A=0)=P(Y=1|A=0)=p$$

$$P(\hat A=0|A=1)=P(Y=0|A=1)=p$$

Entonces:

$$P_e=P_A(0)p+P_A(1)p$$

Sacamos factor comun $p$:

$$P_e=p[P_A(0)+P_A(1)]$$

Como las probabilidades a priori suman uno:

$$P_A(0)+P_A(1)=1$$

queda:

$$P_e=p$$

Para $p=0.3$:

$$\boxed{P_e=0.3}$$

En estos dos puntos, como ML coincide con MAP, esta probabilidad de error es minima.


In [ ]:
def ml_decision(y, p):
    if p < 0.5:
        return y
    if p > 0.5:
        return 1 - y
    return -1  # empate para cualquier y

def map_decision(y, p, q):
    if y == 0:
        score_a0 = q * (1 - p)
        score_a1 = (1 - q) * p
    else:
        score_a0 = q * p
        score_a1 = (1 - q) * (1 - p)

    if score_a0 > score_a1:
        return 0
    if score_a1 > score_a0:
        return 1
    return -1  # empate

for q in [0.6, 0.5]:
    print(f"q = {q}, p = 0.3")
    for y in [0, 1]:
        print(f"  y = {y}: ML decide {ml_decision(y, 0.3)}, MAP decide {map_decision(y, 0.3, q)}")
    print()

## 9. Condicion general para que ML sea optima en este BSC

Para $p<0.5$, ML decide:

$$\hat A_{ML}=Y$$

Para que ML sea optima, debe coincidir con MAP.

Eso significa que MAP tambien debe decidir $A=0$ si $Y=0$ y $A=1$ si $Y=1$.

### Condicion para que, si $y=0$, MAP decida $A=0$

MAP decide $A=0$ si:

$$P_A(0)P_{Y|A}(0|0) \ge P_A(1)P_{Y|A}(0|1)$$

Reemplazamos:

$$q(1-p) \ge (1-q)p$$

Distribuimos el lado derecho:

$$q(1-p) \ge p-pq$$

Distribuimos el lado izquierdo:

$$q-qp \ge p-pq$$

Como $qp=pq$, sumamos $pq$ a ambos lados:

$$q \ge p$$

Primera condicion:

$$q\ge p$$

### Condicion para que, si $y=1$, MAP decida $A=1$

MAP decide $A=1$ si:

$$P_A(1)P_{Y|A}(1|1) \ge P_A(0)P_{Y|A}(1|0)$$

Reemplazamos:

$$(1-q)(1-p) \ge qp$$

Distribuimos el lado izquierdo:

$$1-p-q+qp \ge qp$$

Restamos $qp$ en ambos lados:

$$1-p-q \ge 0$$

Pasamos $q$ al otro lado:

$$1-p \ge q$$

Segunda condicion:

$$q\le 1-p$$

Entonces, para $p<0.5$, la regla ML es optima si:

$$\boxed{p\le q\le 1-p}$$

Para el ejercicio, con $p=0.3$:

$$0.3\le q\le 0.7$$

Por eso tanto $q=0.6$ como $q=0.5$ hacen que ML coincida con MAP.


## 10. Punto 4: proponer valores de $p$ y $q$ para que ML sea optima

Una forma segura de hacer que ML sea optima es elegir hipotesis equiprobables:

$$q=0.5$$

porque entonces:

$$P_A(0)=P_A(1)=0.5$$

y MAP se reduce a ML.

Por ejemplo, podemos proponer:

$$\boxed{p=0.3, \qquad q=0.5}$$

Con esos valores:

$$p=0.3<0.5$$

entonces ML decide:

$$\hat A_{ML}=Y$$

y como $q=0.5$, esa regla tambien es MAP. Por lo tanto, produce la minima probabilidad de error.

Tambien se podria proponer cualquier par que cumpla:

$$p\le q\le 1-p$$

con $p<0.5$.


## 11. Simulacion Monte Carlo de verificacion

Ahora se verifica con una simulacion. Esta parte no reemplaza el desarrollo teorico, solo sirve para comprobar que las cuentas tienen sentido.

Se genera $A$ con probabilidad $P_A(0)=q$ y $P_A(1)=1-q$.

Luego el canal BSC cambia el bit con probabilidad $p$.

Finalmente se aplica la regla ML para $p=0.3$:

$$\hat A=Y$$


In [ ]:
def simulate_bsc_ml(q, p, nb_samples=200_000):
    z_a = rng.uniform(size=nb_samples)
    a = np.where(z_a < q, 0, 1)

    z_error = rng.uniform(size=nb_samples)
    channel_flips = z_error < p
    y = np.where(channel_flips, 1 - a, a)

    if p < 0.5:
        a_hat_ml = y
    elif p > 0.5:
        a_hat_ml = 1 - y
    else:
        a_hat_ml = np.zeros_like(y)

    pe_mc = np.mean(a_hat_ml != a)
    return pe_mc

for q in [0.6, 0.5]:
    pe_mc = simulate_bsc_ml(q=q, p=0.3)
    print(f"q = {q:.1f}, p = 0.3 -> Pe Monte Carlo con ML = {pe_mc:.4f}")

print("Valor teorico esperado para ambos casos: Pe = p = 0.3")

## 12. Respuesta final

Para $q=0.6$ y $p=0.3$, la regla ML es:

$$\boxed{\hat A_{ML}=Y}$$

es decir:

$$\boxed{\hat A=0 \text{ si } Y=0}$$

$$\boxed{\hat A=1 \text{ si } Y=1}$$

Para $q=0.5$ y $p=0.3$, la regla ML es la misma:

$$\boxed{\hat A_{ML}=Y}$$

La razon es que ML solo compara $P_{Y|A}(y|a)$ y no usa las probabilidades a priori.

En los dos casos dados, la regla ML coincide con MAP, por lo tanto es optima y minimiza la probabilidad de error.

La probabilidad de error de esa regla es:

$$\boxed{P_e=p=0.3}$$

Un par de valores que hace que ML sea optima es:

$$\boxed{p=0.3, \qquad q=0.5}$$

Mas generalmente, para $p<0.5$, ML es optima si:

$$\boxed{p\le q\le 1-p}$$
